In [62]:
pip install --upgrade "nudenet>=3.4.2";


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Code

### get_files
"""
Returns a list of image file paths in a directory.

Args:
    folder_path (str): Path to the directory to search for images. Default: "files".
    extensions (tuple): Allowed image file extensions. Default: (".jpeg", ".jpg", ".png").

Returns:
    List[str]: List of image file paths found in the directory.
"""

In [2]:
from pathlib import Path

folder = Path("files")
extensions = (".jpeg", ".jpg", ".png")

files = [
    str(file) 
    for file in folder.iterdir() 
        if 
            file.is_file() and 
            file.suffix.lower() in extensions
]

### detect_nude
"""
Performs batch detection of nudity and explicit content in a list of images using NudeNet.  

Args:  
    files (List[str]): List of image file paths to be analyzed.
    labels (List[str], optional): List of labels to filter the detection results. If not provided, uses the default labels:  
        [  
            "FEMALE_GENITALIA_COVERED", "FEMALE_GENITALIA_EXPOSED",  
            "MALE_GENITALIA_COVERED", "MALE_GENITALIA_EXPOSED",  
            "BUTTOCKS_COVERED", "BUTTOCKS_EXPOSED",  
            "ANUS_EXPOSED", "ANUS_COVERED",  
        ]
  
Returns:  
    List[dict]: A list of dictionaries, each representing an image and its detection results.  
    Each dictionary contains:  
        - 'path': The file path of the image.  
        - 'detect_data': A list of detection dictionaries for the image, filtered by the specified labels.  
            Each detection dictionary includes information such as:  
                - 'class': The detected label/class.  
                - 'score': Confidence score of the detection.  
                - 'box': Bounding box coordinates of the detected region.  
"""

In [ ]:
from nudenet import NudeDetector

valid_labels = [
    "FEMALE_GENITALIA_COVERED", "FEMALE_GENITALIA_EXPOSED",
    "MALE_GENITALIA_COVERED", "MALE_GENITALIA_EXPOSED",
    "ANUS_EXPOSED", "ANUS_COVERED",
]
detector = NudeDetector()
images = detector.detect_batch(files)

nude_results = [
    {
        "name": Path(file).name,
        "path": file,
        "detect_data": [
            detect for detect in image 
                if detect['class'] in valid_labels
        ]
    }
    for file, image in zip(files, images)
]

nude_results

[{'class': 'BUTTOCKS_EXPOSED',
  'score': 0.7513521313667297,
  'box': [239, 807, 492, 295]},
 {'class': 'ANUS_EXPOSED',
  'score': 0.6798909902572632,
  'box': [499, 935, 59, 72]},
 {'class': 'FEMALE_GENITALIA_EXPOSED',
  'score': 0.5208219289779663,
  'box': [464, 997, 137, 141]},
 {'class': 'FACE_FEMALE',
  'score': 0.4443408250808716,
  'box': [178, 185, 186, 199]}]

### 

In [ ]:
import cv2

images = [
    {
        "name": file['name'],
        "path": file['path'],
        "image": {
            "original": cv2.imread(file['path']) 
        },
        "detect_data": file['detect_data']
    } for file in nude_results
]

images

[{'name': '6b100569-dc90-48fe-bbc0-72c581e58a8f.png',
  'path': 'files/6b100569-dc90-48fe-bbc0-72c581e58a8f.png',
  'image': {'original': array([[[ 23,  35,  46],
           [ 18,  26,  41],
           [ 14,  22,  35],
           ...,
           [ 17,  15,  27],
           [ 16,  16,  25],
           [  1,  12,  16]],
   
          [[ 24,  32,  47],
           [ 32,  32,  49],
           [ 24,  26,  41],
           ...,
           [ 24,  20,  33],
           [ 26,  20,  34],
           [ 20,  19,  29]],
   
          [[ 25,  29,  42],
           [ 28,  31,  45],
           [ 24,  25,  39],
           ...,
           [ 25,  20,  33],
           [ 24,  21,  34],
           [ 17,  18,  30]],
   
          ...,
   
          [[198, 178, 170],
           [184, 171, 165],
           [192, 169, 164],
           ...,
           [105, 122, 191],
           [100, 121, 190],
           [ 98, 126, 190]],
   
          [[194, 175, 163],
           [171, 160, 155],
           [182, 164, 160],
      

### censor_image
"""
Applies pixelation censorship to detected regions in an image based on detection data.

Args:
    image (np.ndarray): The original image array to be censored.
    detect_data (List[dict]): List of detection dictionaries, each containing:
        - 'box': Coordinates (x, y, w, h) of the region to censor.
        - 'class': The detected label/class.
        - 'score': Confidence score of the detection.
    pixel_size (int, optional): The size of the pixelation blocks. Default is 3.

Returns:
    np.ndarray: The censored image array with pixelated regions.

Example:
    censored = censor_image(image, detect_data, pixel_size=3)
"""

In [ ]:
first = images[0]

detect_data = first['detect_data']
image = first["image"]["original"].copy()
output_path = "output"
pixel_size = 3

for detect in detect_data:
    x, y, w, h = detect['box']
    roi_small = cv2.resize(
        image[y:y+h, x:x+w], 
        (w // pixel_size, h // pixel_size), 
        interpolation=cv2.INTER_LINEAR
    )
    roi_pixelated = cv2.resize(roi_small, (w, h), interpolation=cv2.INTER_NEAREST)
    image[y:y+h, x:x+w] = roi_pixelated

image
# cv2.imwrite(f"{output_path}/{first["name"]}", image)

True

### add_box_with_padding
"""
Calculates a new bounding box for censorship by adding padding to the detected region.

Args:
    box (List[int]): Original bounding box coordinates [x, y, w, h].
    padding (int, optional): Amount of padding to add around the box. Default is 10.

Returns:
    List[int]: Adjusted bounding box coordinates [x, y, w, h] with padding applied.

Example:
    new_box = add_box_with_padding(box, padding=10)
"""

In [23]:
box = detect_data[0]['box'] # x, y, w, h = detect_data[0]['box']
pixel_size = pixel_size
padding = 10

x, y, w, h = [
    box[0]+padding, 
    box[1]+padding, 
    box[2]-2*padding, 
    box[3]-2*padding
]

[x, y, w, h]

[249, 817, 472, 275]

### add_censor
"""
Applies pixelation censorship to a region in an image based on detection data.

Args:
    box (List[int]): Original bounding box coordinates [x, y, w, h].
    pixel_size (int, optional): The size of the pixelation blocks. Default is 3.

Returns:
    np.ndarray: The censored box with pixelated regions.

Example:
    censored = add_censor(box, pixel_size=3)
"""

In [ ]:
roi_small = cv2.resize(
    image[y:y+h, x:x+w], 
    (w // pixel_size, h // pixel_size), 
    interpolation=cv2.INTER_LINEAR
)
roi_pixelated = cv2.resize(
    roi_small, 
    (w, h), 
    interpolation=cv2.INTER_NEAREST
)
image[y:y+h, x:x+w] = roi_pixelated

### add_debug_square

In [ ]:
square_image = first["image"]["original"].copy()

square_color = (255, 255, 255)
square_thickness = 2
square_x, square_y, square_w, square_h = images[0]['detect_data'][2]['box']


cv2.rectangle(
    square_image, 
    (square_x, square_y, square_w, square_h), 
    square_color, 
    2
)

# cv2.imwrite(f"debug/12.png", square_image)

True

### add_debug_info


In [ ]:
line_color = (255, 255, 255) 
padding_color = (255, 0, 0) 

debug_detect_data = images[0]['detect_data'][2]
debug_image = first["image"]["original"].copy()
debug_name = first["name"]
debug_output_path = "debug"

debug_class = debug_detect_data["class"]
debug_box = debug_detect_data['box']
debug_score = debug_detect_data["score"]

cv2.rectangle(
    debug_image, 
    debug_box, 
    line_color, 
    2
)
cv2.putText(
    debug_image, 
    debug_class,       
    (
        debug_box[0], 
        debug_box[1] - 28
    ), 
    cv2.FONT_HERSHEY_SIMPLEX,
    0.64,
    (255, 255, 255),
    2
)
cv2.putText(
    debug_image, 
    str(debug_score),       
    (
        debug_box[0], 
        debug_box[1] - 8
    ), 
    cv2.FONT_HERSHEY_SIMPLEX,
    0.48,
    (255, 255, 255),
    2
)

padding_line = (
    debug_box[0]+padding, 
    debug_box[1]+padding, 
    debug_box[2]-2*padding, 
    debug_box[3]-2*padding
)
cv2.rectangle(
    debug_image, 
    padding_line, 
    padding_color, 
    1
)

cv2.imwrite(f"{debug_output_path}/{debug_name}", debug_image)

TypeError: list indices must be integers or slices, not str